# Entrevista de Malvinas — ejecución automática y entrega privada

**En el celular:** abrí este notebook en Colab, tocá **Conectar** y después **Entorno de ejecución → Ejecutar todas** (en algunas versiones aparece como **Ejecutar todo**). Autorizá el acceso con la cuenta de Google que puede abrir los originales. Se utiliza una sola autenticación nativa de Colab y Drive API; no se monta DriveFS. No se solicita la URL, un orden manual ni escribir «SI».

Se detectan los tres archivos conocidos y se produce `ENTREVISTA_MALVINAS_FINAL.mp4`, con subtítulos permanentes, y `ENTREVISTA_MALVINAS_SUBTITULOS.srt`. El informe breve y las muestras de las uniones también se guardan en una carpeta privada de Mi unidad. Al finalizar se muestran enlaces reales de Drive, verificados contra tamaño y checksum del servidor.

El orden solicitado y confirmado por el usuario es **≈25 → ≈37 → ≈12 minutos**. Se asignan los tres archivos según las duraciones verificadas con FFprobe. Los checkpoints existentes se reutilizan por identidad del original, y los tiempos del SRT se recalculan para la nueva secuencia. No se eliminan intervenciones ni pausas. La revisión del texto y la sincronía perceptual siguen pendientes.

La ejecución puede tardar, especialmente sin GPU. Cada intervalo transcrito se guarda en Drive y **Ejecutar todas** retoma automáticamente la misma carpeta si se desconecta. Las verificaciones automáticas y las revisiones humanas pendientes se informan por separado. No se usan APIs pagas ni se publica el material.

**Actualizar este notebook no significa que el video real ya se haya generado.** El procesamiento comienza cuando lo ejecutás en tu Colab autenticado.

Si Google muestra un 403 en la ventana `accounts.google.com` antes de completar la autorización, no es un error de transcripción ni de recursos de GPU. Probá un perfil de Chrome con una sola cuenta autorizada. Este notebook no puede cambiar restricciones de cuenta u organización. Si el error sucede después, la celda indica la etapa y el motivo devuelto por Drive API, sin mostrar credenciales.


In [ ]:
# 1 — Dependencias mínimas. No instala Remotion ni herramientas de diseño.
import sys,subprocess,os,shutil
assert 'google.colab' in sys.modules or os.path.isdir('/content'),'Abrí este notebook en Google Colab.'
subprocess.run([sys.executable,'-m','pip','install','-q','faster-whisper==1.2.1','pysubs2==1.8.0','google-api-python-client>=2,<3'],check=True)
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','ffmpeg','fonts-dejavu-core'],check=True)
filters=subprocess.run(['ffmpeg','-hide_banner','-filters'],capture_output=True,text=True,check=True).stdout
assert ' ass ' in filters and 'loudnorm' in filters,'FFmpeg necesita libass y loudnorm.'
os.environ['HF_HOME']='/content/video_studio_hf'
os.environ['HF_XET_CACHE']='/content/video_studio_hf/xet'
print('Dependencias disponibles. El proceso continuará sin preguntas editoriales.')


In [ ]:
"""Self-contained media engine embedded in the private Colab notebook.
No credentials, original media, transcripts or API calls are embedded here.
"""
from pathlib import Path
from fractions import Fraction
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import tempfile
import textwrap

CHECKPOINT_SAVER=None


def run(args):
    p = subprocess.run([str(x) for x in args], capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr[-5000:])
    return p.stdout


def probe(path):
    return json.loads(run(['ffprobe', '-v', 'error', '-show_format', '-show_streams', '-show_data_hash', 'sha256', '-of', 'json', path]))


def checksum(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4*1024**2), b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp.json')
    with open(tmp, 'w') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    if CHECKPOINT_SAVER is not None:
        CHECKPOINT_SAVER(path)


def duration(info):
    return float(info['format']['duration'])


def validate(path, expected=None):
    info = probe(path)
    if duration(info) <= 0:
        raise ValueError('Archivo vacío')
    if expected is not None and abs(duration(info)-expected) > .25:
        raise ValueError(f'Duración inesperada: {duration(info)} vs {expected}')
    run(['ffmpeg', '-v', 'error', '-xerror', '-i', path, '-f', 'null', '-'])
    return info


def video_shape(info):
    v = next(s for s in info['streams'] if s['codec_type']=='video')
    w, h = v['width'], v['height']
    rotations = [s.get('rotation', 0) for s in v.get('side_data_list', [])]
    if any(abs(int(r))%180 == 90 for r in rotations):
        w, h = h, w
    return w, h, v.get('avg_frame_rate', v.get('r_frame_rate', '30/1'))


def compatible_video_copy(details):
    signatures=[]
    for info in details:
        v=next(s for s in info['streams'] if s['codec_type']=='video')
        if v['codec_name']!='h264' or abs(float(v.get('start_time',0)))>.05:
            return False
        if any(int(x.get('rotation',0))%360 for x in v.get('side_data_list',[])):
            return False
        if abs(float(v.get('duration',duration(info)))-duration(info))>.1:
            return False
        signatures.append(tuple(v.get(k) for k in ('codec_name','width','height','pix_fmt',
            'avg_frame_rate','r_frame_rate','sample_aspect_ratio','profile','level','extradata_hash')))
    return bool(signatures) and all(s==signatures[0] for s in signatures)


def normalize(source, target, shape, silence=False, copy_video=False):
    target=Path(target); info=probe(source); seconds=duration(info)
    if target.exists():
        return validate(target,seconds)
    w,h,fps=shape
    partial=target.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    args=['ffmpeg','-v','error','-n','-i',source]
    audio=any(s['codec_type']=='audio' for s in info['streams']) and not silence
    if not audio:
        args+=['-f','lavfi','-i','anullsrc=r=48000:cl=stereo']
    args+=['-map','0:v:0','-map','0:a:0' if audio else '1:a:0']
    if copy_video:
        args+=['-c:v','copy']
    else:
        args+=['-vf',f'scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={fps},tpad=stop_mode=clone:stop_duration={seconds}',
               '-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p']
    args+=['-af','aresample=48000:async=1:first_pts=0,afftdn=nr=6:nf=-35,loudnorm=I=-16:TP=-1.5:LRA=11,aresample=48000,apad',
           '-t',str(seconds),'-c:a','aac','-b:a','128k','-ar','48000','-ac','2','-threads','2',
           '-map_metadata','-1','-movflags','+faststart',partial]
    run(args); validate(partial,seconds); os.replace(partial,target)
    return probe(target)


def join_clips(clips, target):
    target = Path(target)
    expected = sum(duration(probe(p)) for p in clips)
    if target.exists():
        return validate(target, expected)
    partial = target.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as t:
        manifest = Path(t)/'clips.txt'
        manifest.write_text(''.join("file '"+str(Path(p).resolve()).replace("'", "'\\''")+"'\n" for p in clips))
        run(['ffmpeg', '-v', 'error', '-n', '-f', 'concat', '-safe', '0', '-i', manifest,
             '-c', 'copy', '-movflags', '+faststart', partial])
    validate(partial, expected); os.replace(partial,target)
    return probe(target)


def asr_audio_samples(audio):
    """Read the FFmpeg-generated PCM WAV without invoking PyAV's decoder.
    faster-whisper and OpenAI Whisper both accept a 16 kHz float32 NumPy waveform.
    """
    import wave
    import numpy as np
    if isinstance(audio,np.ndarray):
        samples=np.asarray(audio,dtype=np.float32)
        if samples.ndim!=1 or not np.isfinite(samples).all():
            raise ValueError('La señal ASR debe ser mono y contener muestras finitas.')
        return np.ascontiguousarray(samples)
    with wave.open(str(audio),'rb') as wav:
        if (wav.getnchannels(),wav.getframerate(),wav.getsampwidth(),wav.getcomptype())!=(1,16000,2,'NONE'):
            raise ValueError('El intervalo ASR debe ser WAV PCM16 mono a 16000 Hz, generado por FFmpeg.')
        frames=wav.getnframes();raw=wav.readframes(frames)
    if len(raw)!=frames*2:
        raise ValueError('El intervalo WAV está incompleto.')
    return np.frombuffer(raw,dtype='<i2').astype(np.float32)/32768.0


ASR_RUNTIME={}


def load_asr(model_name=None, allow_openai_fallback=True):
    import ctranslate2, gc
    from faster_whisper import WhisperModel
    try:
        available=int(next(x.split()[1] for x in Path('/proc/meminfo').read_text().splitlines() if x.startswith('MemAvailable:')))*1024
    except (OSError,StopIteration,ValueError):
        available=4*1024**3
    gpu=False; compute='int8'
    try:
        gpu=ctranslate2.get_cuda_device_count()>0
        if gpu:
            supported=ctranslate2.get_supported_compute_types('cuda')
            compute='float16' if 'float16' in supported else 'int8_float16'
            gpu=compute in supported
    except (RuntimeError,ValueError):
        gpu=False
    name=model_name or ('small' if gpu or (available>=4*1024**3 and (os.cpu_count() or 1)>=4) else 'base')
    if available<2*1024**3 and not gpu:
        name='base'
    device='cuda' if gpu else 'cpu'
    threads=min(4,os.cpu_count() or 1)
    def cuda_error(error):
        return any(x in str(error).lower() for x in ('cuda','cudnn','cublas','out of memory','insufficient memory'))
    def make_fw(selected):
        return WhisperModel(name,device=selected,compute_type=compute if selected=='cuda' else 'int8',
                           cpu_threads=threads,download_root='/content/video_studio_models')
    ASR_RUNTIME.update(model=name,device=device,engine='faster-whisper',human_review=False)
    try:
        try:
            model=make_fw(device)
        except (RuntimeError,OSError) as error:
            if device!='cuda' or not cuda_error(error):
                raise
            device='cpu'; model=make_fw(device); ASR_RUNTIME['device']=device
        def decode(audio):
            segments,_=model.transcribe(asr_audio_samples(audio),language='es',beam_size=5 if device=='cuda' else 3,
                vad_filter=True,vad_parameters={'threshold':.25,'min_silence_duration_ms':1000,'speech_pad_ms':400},
                word_timestamps=True,condition_on_previous_text=False)
            results=[]
            for s in segments:
                if s.text.strip() and not s.words:
                    raise ValueError('ASR devolvió texto sin marcas temporales de palabra')
                results.append({'text':s.text,'avg_logprob':s.avg_logprob,'no_speech_prob':s.no_speech_prob,
                    'words':[{'word':w.word,'start':w.start,'end':w.end,'probability':w.probability} for w in (s.words or [])]})
            return results
        def infer(audio):
            nonlocal model,device
            try:
                return decode(audio)
            except (RuntimeError,OSError) as error:
                if device!='cuda' or not cuda_error(error):
                    raise
                print('GPU incompatible o sin memoria: se continúa este intervalo en CPU/int8.')
                model=None; gc.collect(); device='cpu'; model=make_fw(device)
                ASR_RUNTIME['device']=device
                return decode(audio)
        return infer,'faster-whisper:'+name
    except Exception as error:
        if not allow_openai_fallback:
            raise
        print('Faster-whisper no se pudo cargar:',type(error).__name__)
        print('Se intenta OpenAI Whisper local, con pesos oficiales y SHA-256 de su cargador. Sin API paga.')
        run([os.sys.executable,'-m','pip','install','-q','openai-whisper==20250625'])
        import whisper
        try:
            import torch
            device='cuda' if torch.cuda.is_available() else 'cpu'
        except ImportError:
            device='cpu'
        try:
            model=whisper.load_model(name,device=device,download_root='/content/video_studio_models')
        except (RuntimeError,OSError) as error:
            if device!='cuda' or not cuda_error(error):
                raise
            device='cpu';model=whisper.load_model(name,device=device,download_root='/content/video_studio_models')
        ASR_RUNTIME.update(engine='openai-whisper',device=device)
        def infer(audio):
            nonlocal model,device
            try:
                result=model.transcribe(asr_audio_samples(audio),language='es',fp16=device=='cuda',verbose=False,
                                        word_timestamps=True,condition_on_previous_text=False)
            except (RuntimeError,OSError) as error:
                if device!='cuda' or not cuda_error(error):
                    raise
                model=None;gc.collect();device='cpu'
                model=whisper.load_model(name,device=device,download_root='/content/video_studio_models')
                ASR_RUNTIME['device']=device
                result=model.transcribe(asr_audio_samples(audio),language='es',fp16=False,verbose=False,
                                        word_timestamps=True,condition_on_previous_text=False)
            return result['segments']
        return infer,'openai-whisper:'+name


def transcribe_clip(source, source_key, checkpoint_root, infer, engine, chunk=300):
    """2 s context around each owned interval; assign words by midpoint.
    Saves each completed interval through the authenticated Drive API.
    """
    checkpoint_root = Path(checkpoint_root); checkpoint_root.mkdir(parents=True, exist_ok=True)
    seconds = duration(probe(source)); words = []; flags = []
    for i in range(math.ceil(seconds/chunk)):
        owned_start, owned_end = i*chunk, min((i+1)*chunk, seconds)
        start, end = max(0, owned_start-2), min(seconds, owned_end+2)
        key = {'version':3, 'source':source_key, 'engine':engine, 'interval':[owned_start, owned_end]}
        record = checkpoint_root/f'{i:04d}.json'
        reused=record.exists()
        if not reused and globals().get('CHECKPOINTS_ONLY',False):
            raise RuntimeError(f'Falta el checkpoint {record.name} de {checkpoint_root.name}; no se reinició la transcripción.')
        if reused:
            data = json.loads(record.read_text())
            if data['key'] != key:
                raise ValueError('Checkpoint incompatible. Usar otra carpeta de trabajo.')
        else:
            with tempfile.TemporaryDirectory() as t:
                audio = Path(t)/'audio.wav'
                run(['ffmpeg', '-v', 'error', '-n', '-ss', str(start), '-i', source,
                     '-t', str(end-start), '-vn', '-ac', '1', '-ar', '16000', '-c:a', 'pcm_s16le', audio])
                results = infer(audio)
            chosen = []; warnings = []
            for s in results:
                if s.get('avg_logprob', 0)<-.8 or s.get('no_speech_prob', 0)>.5:
                    warnings.append({'start':start+s['words'][0]['start'] if s.get('words') else start,
                                     'text':s.get('text',''), 'reason':'revisar_confianza'})
                for word in s.get('words', []):
                    a, b = start+float(word['start']), start+float(word['end'])
                    if owned_start <= (a+b)/2 < owned_end:
                        chosen.append({'word':word['word'], 'start':max(0,a), 'end':min(seconds,b),
                                       'probability':word.get('probability')})
            data = {'key':key, 'words':chosen, 'review':warnings}
            atomic_json(record, data)
        words += data['words']; flags += data['review']
        status='RECUPERADO, sin transcribir' if reused else 'guardado'
        print(f'{Path(source).name}: intervalo {i+1}/{math.ceil(seconds/chunk)} {status}')
    return words, flags


def cues_from_words(words, width=1920, height=1080):
    line_limit=max(18,min(42,round(42*(width/height)/(16/9))))
    # A recognizer can return several whitespace-separated words in one timed token.
    # Split only that token for layout; its original interval and text are preserved.
    units=[]
    for original in words:
        pieces=original['word'].split()
        if not pieces:continue
        if len(pieces)==1:
            units.append({**original,'word':pieces[0]})
        else:
            total=sum(len(piece) for piece in pieces);cursor=0
            start,end=float(original['start']),float(original['end'])
            for piece in pieces:
                a=start+(end-start)*cursor/total;cursor+=len(piece)
                b=start+(end-start)*cursor/total
                units.append({**original,'word':piece,'start':a,'end':b})
    def text(group):
        return re.sub(r'\s+([,.;:!?])',r'\1',' '.join(w['word'].strip() for w in group)).strip()
    def wrapped(group):
        return textwrap.wrap(text(group),line_limit,break_long_words=False,break_on_hyphens=False)
    groups=[];current=[]
    for word in units:
        # Character count alone can wrap onto THREE lines at word boundaries.
        # Test the actual wrapped candidate before adding the next timed word.
        if current and (len(wrapped(current+[word]))>2 or
                        word['end']-current[0]['start']>5 or
                        word['start']-current[-1]['end']>.7 or
                        (text(current).endswith(('.', '?', '!', ';')) and
                         current[-1]['end']-current[0]['start']>1.2)):
            groups.append(current);current=[]
        current.append(word)
    if current:groups.append(current)
    cues=[]
    for group in groups:
        start,end=max(0,group[0]['start']),max(w['end'] for w in group)
        lines=wrapped(group)
        if cues:start=max(start,cues[-1]['end'])
        end=max(start+.05,end)
        cues.append({'start':start,'end':end,'text':'\n'.join(lines)})
    return cues


def write_subtitles(cues, srt, ass, width, height):
    import pysubs2
    subs = pysubs2.SSAFile()
    for cue in cues:
        subs.append(pysubs2.SSAEvent(start=round(cue['start']*1000), end=round(cue['end']*1000),
                                   text=cue['text'].replace('\n',r'\N')))
    subs.info.update(PlayResX=str(width), PlayResY=str(height), WrapStyle='0')
    subs.styles['Default'] = pysubs2.SSAStyle(fontname='DejaVu Sans', fontsize=max(14,48*height/1080),
        primarycolor=pysubs2.Color(255,255,255), outlinecolor=pysubs2.Color(0,0,0),
        outline=max(1.2,2.5*height/1080), shadow=0, alignment=pysubs2.Alignment.BOTTOM_CENTER,
        marginl=round(width*.07), marginr=round(width*.07), marginv=round(height*.06))
    subs.save(str(srt)); subs.save(str(ass))


def export_burned(joined, ass, final):
    final = Path(final)
    if final.exists():
        return validate(final, duration(probe(joined)))
    partial = final.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as t:
        safe = Path(t)/'captions.ass'; shutil.copy2(ass,safe)
        run(['ffmpeg','-v','error','-n','-i',joined,'-vf',f'ass={safe}',
             '-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p',
             '-c:a','copy','-threads','2','-map_metadata','-1','-movflags','+faststart',partial])
    validate(partial, duration(probe(joined))); os.replace(partial,final)
    return probe(final)


def frame(path, at, target):
    run(['ffmpeg','-v','error','-y','-ss',str(at),'-i',path,'-frames:v','1',target])


def pipeline(sources, local_root, durable_root, infer, engine, intro=None, reuse=None):
    """sources are automatically ordered; originals remain untouched; human review stays pending."""
    local_root = Path(local_root); durable_root = Path(durable_root)
    local_root.mkdir(parents=True,exist_ok=True); durable_root.mkdir(parents=True,exist_ok=True)
    source_hashes = [checksum(p) for p in sources]
    details = [probe(p) for p in sources]
    if len(sources)!=3 or len(set(source_hashes))!=3:
        raise ValueError('Se requieren tres archivos distintos')
    shape = tuple(reuse['shape']) if reuse else video_shape(details[0])
    shape = (shape[0]+shape[0]%2, shape[1]+shape[1]%2, shape[2])
    if shape[2] in ('0/0','0/1'):
        shape = (shape[0],shape[1],'30/1')
    intro_hash = checksum(intro) if intro else None
    key = {'sources':source_hashes,'shape':list(shape),'intro':intro_hash,'engine':engine,'version':3}
    lockfile = durable_root/'job.json'
    if lockfile.exists() and json.loads(lockfile.read_text())!=key:
        raise ValueError('Otro orden/configuración ya usa esta carpeta. Elegir una nueva carpeta de trabajo.')
    atomic_json(lockfile,key)
    required = sum(Path(p).stat().st_size for p in sources)*4 + 1024**3
    if shutil.disk_usage(local_root).free < required:
        raise RuntimeError(f'Espacio local insuficiente. Estimación mínima: {required/1024**3:.1f} GiB')
    copy_video=compatible_video_copy(details)
    normalized = []; rows = []; all_words = []; review = []; offset = 0
    if intro:
        p = local_root/'intro.mp4'; d = normalize(intro,p,shape,silence=True)
        normalized.append(p); offset = duration(d)
    for i, (source, info, sha) in enumerate(zip(sources,details,source_hashes)):
        asset=reuse['assets'][sha] if reuse else None
        cached=Path(asset['normalized']) if asset else None
        if cached is not None and cached.exists():
            target=cached;d=validate(target,duration(info))
            print(f'VIDEO REUTILIZADO: {Path(source).name}, {duration(d)/60:.2f} min')
        else:
            target=local_root/f'normalized_{i}.mp4'
            d=normalize(source,target,shape,copy_video=copy_video)
        normalized.append(target)
        rows.append({'source':Path(source).name,'sha256':sha,'source_duration':duration(info),
                     'output_start':offset,'output_end':offset+duration(d)})
        checkpoint_root=Path(asset['checkpoints']) if asset else durable_root/f'checkpoints_{i}'
        words, warnings = transcribe_clip(target,sha,checkpoint_root,infer,engine)
        all_words += [{**w,'start':w['start']+offset,'end':w['end']+offset} for w in words]
        review += [{**w,'start':w['start']+offset} for w in warnings]
        offset += duration(d)
    joined = local_root/'joined.mp4'; join_clips(normalized,joined)
    cues = cues_from_words(all_words,*shape[:2])
    if not cues:
        raise ValueError('No hay subtítulos reconocidos; no se exporta una entrevista muda de texto')
    if cues[-1]['end']>duration(probe(joined))+.1:
        raise ValueError('Subtítulos fuera de la duración')
    srt = local_root/'ENTREVISTA_MALVINAS_SUBTITULOS.srt'
    ass = local_root/'captions.ass'; write_subtitles(cues,srt,ass,*shape[:2])
    shutil.copy2(srt,durable_root/srt.name)
    atomic_json(durable_root/'words.json',all_words)
    final = local_root/'ENTREVISTA_MALVINAS_FINAL.mp4'; final_info = export_burned(joined,ass,final)
    streams = final_info['streams']
    if next(s for s in streams if s['codec_type']=='video')['codec_name']!='h264':
        raise ValueError('Codec de video incorrecto')
    if next(s for s in streams if s['codec_type']=='audio')['codec_name']!='aac':
        raise ValueError('Codec de audio incorrecto')
    for i, cue in enumerate(cues):
        cps = len(cue['text'].replace('\n',''))/(cue['end']-cue['start'])
        if cps>22:
            review.append({'cue':i+1,'start':cue['start'],'reason':'velocidad_lectura','cps':round(cps,1)})
    report = {'clips':rows,'intro_seconds':rows[0]['output_start'],
              'expected_seconds':sum(duration(probe(p)) for p in normalized),
              'final_seconds':duration(final_info),'decode_complete':True,
              'subtitles_count':len(cues),'human_sync_review':False,'human_editorial_review':False,
              'preliminary_video_stream_copy':copy_video,'review':review,
              'reused_source_checkpoints':bool(reuse),
              'final_sha256':checksum(final),'srt_sha256':checksum(srt)}
    # Difference test and saved frames at spread-out, actually captioned times.
    import numpy as np
    shots = []; seen = set()
    for index in [0,len(cues)//4,len(cues)//2,3*len(cues)//4,len(cues)-1]:
        cue = cues[index]; at = (cue['start']+cue['end'])/2
        if index in seen: continue
        seen.add(index)
        def pixels(path):
            return subprocess.run(['ffmpeg','-v','error','-ss',str(at),'-i',str(path),
               '-frames:v','1','-f','rawvideo','-pix_fmt','gray','-'],check=True,capture_output=True).stdout
        before = np.frombuffer(pixels(joined),dtype=np.uint8).astype(np.int16)
        after = np.frombuffer(pixels(final),dtype=np.uint8).astype(np.int16)
        changed = int(np.count_nonzero(np.abs(after-before)>35))
        if changed < 10:
            raise ValueError('No se detectó texto quemado en el fotograma esperado')
        shot = local_root/f'check_{index:04d}.png'; frame(final,at,shot)
        shots.append({'at':at,'file':str(shot),'cue':cue['text'],'changed_pixels':changed})
    report['subtitle_frame_checks']=shots
    atomic_json(durable_root/'validation.json',report)
    # Save recoverable text immediately, before uploading the large final file.
    shutil.copy2(srt,durable_root/srt.name)
    atomic_json(durable_root/'words.json',all_words)
    return final,srt,report


def automatic_order(paths):
    from itertools import permutations
    if len(paths)!=3:raise ValueError('Se requieren los tres originales conocidos.')
    info={str(p):probe(p) for p in paths}
    requested=(25*60,37*60,12*60)
    choices=[]
    for order in permutations(paths):
        differences=[abs(duration(info[str(path)])-target) for path,target in zip(order,requested)]
        choices.append((sum(differences),tuple(order),differences))
    choices.sort(key=lambda row:row[0])
    cost,order,differences=choices[0]
    if max(differences)>180 or abs(choices[1][0]-cost)<1:
        raise ValueError('Las duraciones no identifican de forma única los archivos de aproximadamente 25, 37 y 12 minutos. No se cambia el orden por suposición.')
    evidence=[{'source':Path(path).name,'duration':duration(info[str(path)]),
               'minutes':round(duration(info[str(path)])/60,3),'shape':video_shape(info[str(path)]),
               'codecs':[stream.get('codec_name') for stream in info[str(path)]['streams']]} for path in order]
    report={'order':[Path(path).name for path in order],
        'method':'orden indicado por el usuario: aproximadamente 25 → 37 → 12 minutos; archivos identificados por FFprobe',
        'requested_minutes':[25,37,12],'actual_minutes':[row['minutes'] for row in evidence],
        'order_confirmed_by_user':True,'reviewed_by_user':True,'human_review_scope':'solo el orden solicitado',
        'semantic_continuity_verified':False,
        'status':'orden confirmado por el usuario; revisión de texto y sincronía pendiente',
        'evidence':evidence}
    return list(order),report


def prepare_reorder_reuse(sources,base_local,base_work,engine):
    base_local=Path(base_local);base_work=Path(base_work)
    job=base_work/'job.json'
    if not job.exists():return None
    previous=json.loads(job.read_text())
    wanted=[checksum(path) for path in sources]
    old=previous['sources']
    if set(wanted)!=set(old) or len(set(old))!=3:
        raise ValueError('Los originales no coinciden con el trabajo guardado. No se mezclan transcripciones de otros archivos.')
    if previous['engine']!=engine:
        raise ValueError('El motor de los checkpoints no coincide. No se inicia una nueva transcripción.')
    if wanted==old:return None
    assets={sha:{'normalized':str(base_local/f'normalized_{i}.mp4'),
                 'checkpoints':str(base_work/f'checkpoints_{i}')} for i,sha in enumerate(old)}
    # A distinct namespace prevents an existing joined/final MP4 from reusing the old order.
    suffix='orden_25_37_12_'+hashlib.sha256('|'.join(wanted).encode()).hexdigest()[:10]
    return {'shape':previous['shape'],'assets':assets,'namespace':suffix,
            'old_source_order':old,'new_source_order':wanted}


In [ ]:
# 2 — Una sola autenticación nativa de Colab. No se monta DriveFS.
# Si Google rechaza el consentimiento, no se intenta eludir su control de acceso.
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
import google.auth
from datetime import datetime
from pathlib import PurePosixPath
from googleapiclient.errors import HttpError

FOLDER_ID='1ps24ehNBibgSXvSNF-h8JSz2MYZv4BGX'
EXPECTED=['VID-20261107-WA0034.mp4','VID-20261107-WA0035.mp4','VID-20261107-WA0036.mp4']
print('AUTORIZACIÓN: Colab solicitará acceso con tu cuenta de Google.')
try:
    auth.authenticate_user()
    credentials,_=google.auth.default()
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
except Exception as error:
    raise RuntimeError('No se completó la autorización de Google en Colab. Si la ventana accounts.google.com muestra 403, probá un perfil de Chrome con una sola cuenta que tenga acceso a los videos. No cambies los archivos a públicos ni compartas tokens.') from error

def explain_drive_error(error,stage):
    status=getattr(getattr(error,'resp',None),'status',None)
    reason='sin detalle'
    try:
        data=json.loads(error.content.decode()).get('error',{})
        reasons=[x.get('reason','') for x in data.get('errors',[])]
        reasons += [x.get('reason','') for x in data.get('details',[]) if isinstance(x,dict)]
        reason=', '.join(x for x in reasons if x) or data.get('status','sin detalle')
    except (ValueError,AttributeError,UnicodeError):
        pass
    help_text=''
    if status==403 and any(x in reason for x in ('insufficientPermissions','ACCESS_TOKEN_SCOPE_INSUFFICIENT')):
        help_text=' La autorización no incluye el acceso requerido a Drive; volver a autorizar Colab con la cuenta correcta.'
    elif status==403 and any(x in reason.lower() for x in ('domain','admin','policy','restricted')):
        help_text=' Google indica una restricción de la cuenta u organización; corresponde revisar esa política con su administrador.'
    elif status in (403,404):
        help_text=' Confirmá que la misma cuenta autorizada puede abrir la carpeta fuente en drive.google.com. Un 404 también puede indicar falta de acceso, no necesariamente que la carpeta no exista.'
    return RuntimeError(f'Drive API: HTTP {status}; etapa {stage}; motivo {reason}.'+help_text)

# Se distingue el rechazo de Google en el consentimiento de un error posterior de la API.
try:
    source_folder=service.files().get(fileId=FOLDER_ID,supportsAllDrives=True,fields='id,mimeType').execute()
    assert source_folder['mimeType']=='application/vnd.google-apps.folder','El ID fuente no identifica una carpeta.'
except HttpError as error:
    raise explain_drive_error(error,'comprobar acceso a la carpeta fuente') from None
print('AUTORIZACIÓN COMPLETA: la API puede leer la carpeta fuente.')
LOCAL=Path('/content/video_studio_malvinas');LOCAL.mkdir(exist_ok=True)
INPUT=LOCAL/'input';INPUT.mkdir(exist_ok=True)
items=[];token=None
while True:
    page=service.files().list(q=f"'{FOLDER_ID}' in parents and trashed=false",pageToken=token,
        pageSize=1000,includeItemsFromAllDrives=True,supportsAllDrives=True,
        fields='nextPageToken,files(id,name,size,md5Checksum,mimeType,capabilities/canDownload)').execute()
    items+=page.get('files',[]);token=page.get('nextPageToken')
    if not token:break
selected=[]
for name in EXPECTED:
    matches=[f for f in items if f['name']==name]
    assert len(matches)==1,f'Bloqueo real: falta el original o hay duplicados de {name}.'
    info=matches[0]
    assert info.get('capabilities',{}).get('canDownload',True),f'Sin permiso de descarga: {name}'
    assert int(info.get('size',0))>0,f'Archivo vacío/no descargable: {name}'
    selected.append(info)

# La misma versión de los originales recupera sus checkpoints, sin pedir nombres o rutas.
fingerprint=hashlib.sha256(json.dumps([(f['id'],f['size'],f.get('md5Checksum')) for f in selected]).encode()).hexdigest()[:20]
JOB_TAG='malvinas_auto_v3_'+fingerprint
query="'root' in parents and trashed=false and mimeType='application/vnd.google-apps.folder' and appProperties has { key='videoStudioJob' and value='"+JOB_TAG+"' }"
candidates=service.files().list(q=query,fields='files(id,name)',orderBy='createdTime desc',pageSize=100).execute().get('files',[])
def private_owner_only(file_id):
    permissions=service.permissions().list(fileId=file_id,fields='permissions(type,role)').execute().get('permissions',[])
    return bool(permissions) and all(p['type']=='user' and p['role']=='owner' for p in permissions)
remote_folder=next((f for f in candidates if private_owner_only(f['id'])),None)
if remote_folder is None:
    name='VideoStudioAI_Malvinas_'+datetime.now().strftime('%Y%m%d_%H%M%S')
    remote_folder=service.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder',
        'appProperties':{'videoStudioJob':JOB_TAG}},fields='id,name').execute()
assert private_owner_only(remote_folder['id']),'Bloqueo real: no se pudo disponer de una carpeta privada.'
# Caché local restaurada desde Drive; no necesita DriveFS ni un segundo consentimiento.
DURABLE=LOCAL/'checkpoint_cache'/fingerprint
DURABLE.mkdir(parents=True,exist_ok=True)


def upload_private_verified(path,mimetype,remote_name=None,checkpoint_relative=None):
    path=Path(path);name=remote_name or path.name
    escaped=name.replace('\\','\\\\').replace("'","\\'")
    matches=service.files().list(q=f"'{remote_folder['id']}' in parents and name='{escaped}' and trashed=false",
        fields='files(id,size,md5Checksum)',pageSize=100).execute().get('files',[])
    assert len(matches)<=1,'Hay copias ambiguas de un archivo generado; no se modifica ninguna sin identificarlas.'
    local_md5=checksum(path,'md5');result=None
    if matches and matches[0].get('md5Checksum')==local_md5 and int(matches[0]['size'])==path.stat().st_size:
        result=matches[0]
    else:
        media=MediaFileUpload(str(path),mimetype=mimetype,chunksize=8*1024**2,resumable=True)
        if matches:
            request=service.files().update(fileId=matches[0]['id'],media_body=media,fields='id')
        else:
            body={'name':name,'parents':[remote_folder['id']]}
            if checkpoint_relative is not None:body['appProperties']={'vsaiRelPath':checkpoint_relative}
            request=service.files().create(body=body,media_body=media,fields='id')
        while result is None:
            status,result=request.next_chunk(num_retries=3)
            if status and checkpoint_relative is None:print(name,round(status.progress()*100),'% cargado')
    stored=service.files().get(fileId=result['id'],fields='id,name,size,md5Checksum,webViewLink').execute()
    assert int(stored['size'])==path.stat().st_size,'Drive no confirmó el tamaño completo.'
    assert stored['md5Checksum']==local_md5,'Drive no confirmó el checksum.'
    assert private_owner_only(stored['id']),'El archivo generado no está exclusivamente bajo el propietario.'
    return stored


def safe_checkpoint_path(relative):
    rel=PurePosixPath(relative)
    if rel.is_absolute() or '..' in rel.parts or not rel.parts or rel.suffix!='.json':
        raise ValueError('Ruta de checkpoint no admitida.')
    allowed={'source_manifest.json','order_report.json','model_selection.json','entrega_verificada.json'}
    if rel.parts[0] not in allowed and not rel.parts[0].startswith('trabajo_'):
        raise ValueError('El archivo no pertenece al trabajo recuperable.')
    return DURABLE.joinpath(*rel.parts)


def restore_checkpoints(folder_id,prefix='',depth=0):
    if depth>4:raise ValueError('Estructura de checkpoints demasiado profunda.')
    restored=0;token=None
    while True:
        page=service.files().list(q=f"'{folder_id}' in parents and trashed=false",pageToken=token,pageSize=1000,
            fields='nextPageToken,files(id,name,mimeType,size,md5Checksum,appProperties)').execute()
        for item in page.get('files',[]):
            name=item['name']
            if item['mimeType']=='application/vnd.google-apps.folder':
                # También recupera los checkpoints del notebook anterior, guardados con DriveFS.
                if (not prefix and name.startswith('trabajo_')) or (prefix.startswith('trabajo_') and name.startswith('checkpoints_')):
                    restored+=restore_checkpoints(item['id'],prefix+name+'/',depth+1)
                continue
            relative=item.get('appProperties',{}).get('vsaiRelPath') or (prefix+name)
            try:target=safe_checkpoint_path(relative)
            except ValueError:continue
            # Las nuevas copias de la API prevalecen sobre el árbol antiguo de DriveFS.
            if target.exists() and not item.get('appProperties',{}).get('vsaiRelPath'):continue
            blob=service.files().get_media(fileId=item['id']).execute(num_retries=3)
            assert len(blob)==int(item['size']),'Checkpoint incompleto en Drive.'
            if item.get('md5Checksum'):assert hashlib.md5(blob).hexdigest()==item['md5Checksum'],'Checksum de checkpoint incorrecto.'
            json.loads(blob) # comprobar formato antes de restaurar
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_suffix('.restore.tmp');temp.write_bytes(blob);os.replace(temp,target)
            restored+=1
        token=page.get('nextPageToken')
        if not token:break
    return restored


def persist_checkpoint(path):
    try:relative=Path(path).resolve().relative_to(DURABLE.resolve()).as_posix()
    except ValueError:return
    safe_checkpoint_path(relative)
    remote_name='checkpoint__'+relative.replace('/','__')
    try:upload_private_verified(path,'application/json',remote_name,relative)
    except HttpError as error:raise explain_drive_error(error,'guardar checkpoint privado') from None

restored=restore_checkpoints(remote_folder['id'])
CHECKPOINT_SAVER=persist_checkpoint
print('Checkpoints recuperados desde Drive:',restored)
print('Carpeta privada y recuperable: https://drive.google.com/drive/folders/'+remote_folder['id'])
required=sum(int(f['size']) for f in selected)*5+1024**3
assert shutil.disk_usage(LOCAL).free>required,'Bloqueo real: espacio local insuficiente.'
originals=[]
for info in selected:
    path=INPUT/info['name']
    valid=path.exists() and path.stat().st_size==int(info['size'])
    if valid and info.get('md5Checksum'):valid=checksum(path,'md5')==info['md5Checksum']
    if not valid:
        partial=path.with_suffix('.partial');partial.unlink(missing_ok=True)
        with open(partial,'wb') as f:
            downloader=MediaIoBaseDownload(f,service.files().get_media(fileId=info['id'],supportsAllDrives=True),chunksize=16*1024**2)
            done=False
            while not done:
                progress,done=downloader.next_chunk(num_retries=3)
                print(info['name'],round(progress.progress()*100),'% descargado')
        assert partial.stat().st_size==int(info['size']),'Tamaño descargado incorrecto.'
        if info.get('md5Checksum'):assert checksum(partial,'md5')==info['md5Checksum'],'Checksum de origen incorrecto.'
        validate(partial);os.replace(partial,path)
    originals.append(path)
atomic_json(DURABLE/'source_manifest.json',selected)
print('Tres copias verificadas. Los originales remotos no se modifican.')


In [ ]:
# 3 — Orden confirmado: aproximadamente 25 → 37 → 12 minutos, comprobado con FFprobe.
sources,ORDER_REPORT=automatic_order(originals)
atomic_json(DURABLE/'order_report.json',ORDER_REPORT)
print('Orden:',ORDER_REPORT['order'])
print('Criterio:',ORDER_REPORT['method'])
print('Duraciones reales, minutos:',ORDER_REPORT['actual_minutes'])
print('Se aplica el orden indicado por el usuario. La revisión editorial continúa pendiente.')
# No se crea ninguna apertura nueva: la prioridad son los tres originales completos.
INTRO=None


In [ ]:
# 4 — Transcripción, unión y subtítulos permanentes. GPU compatible o CPU eficiente.
MODEL_RECORD=DURABLE/'model_selection.json'
_previous_recovery_flag=bool(globals().get('CHECKPOINTS_ONLY',False))
# A completed previous delivery proves that ASR checkpoints should already exist.
CHECKPOINTS_ONLY=_previous_recovery_flag or (DURABLE/'entrega_verificada.json').exists()
if CHECKPOINTS_ONLY:
    if not MODEL_RECORD.exists():
        raise RuntimeError('Falta el registro del modelo. No se reinició ninguna transcripción.')
    saved_runtime=json.loads(MODEL_RECORD.read_text())
    backend=saved_runtime.get('engine');model=saved_runtime.get('model')
    if backend not in ('faster-whisper','openai-whisper') or model not in ('base','small'):
        raise RuntimeError('No se pudo identificar el motor de los checkpoints; no se reinició la transcripción.')
    engine=backend+':'+model
    ASR_RUNTIME.update(saved_runtime)
    def infer(audio):
        raise RuntimeError('Falta un intervalo guardado. El modo de recuperación no vuelve a transcribir; conserva los avances existentes.')
    print('RECUPERACIÓN: solo checkpoints existentes. Whisper no se carga ni se ejecuta.')
else:
    preferred=json.loads(MODEL_RECORD.read_text()).get('model') if MODEL_RECORD.exists() else None
    if preferred not in ('base','small'):preferred=None
    infer,engine=load_asr(model_name=preferred,allow_openai_fallback=True)
    atomic_json(MODEL_RECORD,dict(ASR_RUNTIME))
print('Motor seleccionado:',engine,ASR_RUNTIME)
ENGINE_KEY=re.sub(r'[^A-Za-z0-9_-]','_',engine)
BASE_WORK=DURABLE/('trabajo_'+ENGINE_KEY)
BASE_LOCAL=LOCAL/'render'/fingerprint/ENGINE_KEY
REORDER_REUSE=prepare_reorder_reuse(sources,BASE_LOCAL,BASE_WORK,engine)
WORK=BASE_WORK/REORDER_REUSE['namespace'] if REORDER_REUSE else BASE_WORK
RENDER=BASE_LOCAL/REORDER_REUSE['namespace'] if REORDER_REUSE else BASE_LOCAL
if REORDER_REUSE:
    print('CORRECCIÓN DE ORDEN: se conservan los checkpoints originales y se recalculan sus tiempos.')
try:
    FINAL,SRT,REPORT=pipeline(sources,RENDER,WORK,infer,engine,intro=INTRO,reuse=REORDER_REUSE)
finally:
    CHECKPOINTS_ONLY=_previous_recovery_flag
REPORT['continuity']=ORDER_REPORT
REPORT['asr_runtime']=dict(ASR_RUNTIME)
REPORT['human_sync_review']=False
REPORT['human_editorial_review']=False
REPORT['automatic_checks']={'three_sources_included':len(REPORT['clips'])==3,
    'decode_complete':REPORT['decode_complete'],
    'duration_matches_sum':abs(REPORT['final_seconds']-REPORT['expected_seconds'])<=.25,
    'subtitle_overlay_frame_checks':bool(REPORT['subtitle_frame_checks']),
    'drive_source_size_checked':True,'drive_source_md5_checked':all(f.get('md5Checksum') for f in selected)}
REPORT['human_reviews_pending']=['continuidad semántica entre grabaciones','sincronía perceptual de voz y subtítulos',
    'nombres propios, fechas y lugares','fidelidad de reconocimiento y naturalidad de voces','descarga efectiva en el celular']
# Contexto textual en cada unión: solo palabras reconocidas, sin corregir ni inventar referencias.
words=json.loads((WORK/'words.json').read_text())
REPORT['boundary_contexts']=[]
for clip in REPORT['clips'][1:]:
    at=clip['output_start']
    REPORT['boundary_contexts'].append({'at':at,
        'before':' '.join(w['word'].strip() for w in words if at-12<=w['start']<at),
        'after':' '.join(w['word'].strip() for w in words if at<=w['start']<at+12)})
# Muestras reales de las uniones, para revisar después; no bloquean la entrega si fallan.
REVIEW_SAMPLES=[];REPORT['sample_warnings']=[]
for i,clip in enumerate(REPORT['clips'][1:],1):
    at=clip['output_start'];sample=LOCAL/f'union_{i}_revision.mp4'
    try:
        sample.unlink(missing_ok=True)
        run(['ffmpeg','-v','error','-n','-ss',str(max(0,at-5)),'-i',FINAL,'-t','10',
             '-vf','scale=640:-2','-c:v','libx264','-preset','veryfast','-crf','26',
             '-c:a','aac','-b:a','96k','-threads','2',sample])
        validate(sample);REVIEW_SAMPLES.append(sample)
    except Exception as error:
        REPORT['sample_warnings'].append({'union':i,'error_type':type(error).__name__})
atomic_json(WORK/'validation.json',REPORT)
print('MP4 decodificado completamente:',REPORT['final_seconds'],'s. Subtítulos:',REPORT['subtitles_count'])
print('Continúa automáticamente la carga en Drive. No se requiere escribir SI.')


In [ ]:
# 5 — Entrega automática: archivos privados, verificación de servidor y enlaces móviles.
assert private_owner_only(remote_folder['id']),'No se carga material en una carpeta compartida.'
def upload_verified(path,mimetype):
    try:return upload_private_verified(path,mimetype)
    except HttpError as error:raise explain_drive_error(error,'cargar entregable final') from None

links=[]
for path,mime in [(FINAL,'video/mp4'),(SRT,'application/x-subrip')]:
    stored=upload_verified(path,mime);links.append(stored)
    print('CARGA VERIFICADA:',stored['name'],stored['webViewLink'])
REPORT['drive_uploads_verified']=[{'name':x['name'],'bytes':int(x['size']),'md5':x['md5Checksum'],'url':x['webViewLink']} for x in links]
REPORT['review_sample_links']=[]
for sample in REVIEW_SAMPLES:
    try:
        stored=upload_verified(sample,'video/mp4');REPORT['review_sample_links'].append(stored['webViewLink'])
    except Exception as error:
        REPORT['sample_warnings'].append({'sample':sample.name,'upload_error_type':type(error).__name__})
report_path=LOCAL/'ENTREVISTA_MALVINAS_INFORME.txt'
lines=['ENTREVISTA DE MALVINAS — INFORME AUTOMÁTICO','',
    'Originales completos: '+', '.join(ORDER_REPORT['order']),
    'Orden: '+ORDER_REPORT['method'],
    'Continuidad semántica: pendiente de revisión humana.',
    f"Duración final: {REPORT['final_seconds']:.3f} s; esperada: {REPORT['expected_seconds']:.3f} s.",
    'Verificaciones automáticas: '+json.dumps(REPORT['automatic_checks'],ensure_ascii=False),
    'Motor local: '+json.dumps(REPORT['asr_runtime'],ensure_ascii=False),
    'MP4 y SRT: carga completa, tamaños y checksum MD5 verificados en Drive.',
    'Revisión humana: NO realizada. Pendiente: '+', '.join(REPORT['human_reviews_pending']),
    'Advertencias ASR/lectura: '+str(len(REPORT['review'])),
    'Checkpoints JSON privados guardados y comprobados mediante Drive API.','',
    'Resultados:']
lines += [x['name']+': '+x['webViewLink'] for x in links]
lines += ['Muestras de continuidad para revisar después:']+REPORT['review_sample_links']
if REPORT['sample_warnings']:lines+=['Advertencias de muestras: '+json.dumps(REPORT['sample_warnings'],ensure_ascii=False)]
report_path.write_text('\n'.join(lines),encoding='utf-8')
report_link=upload_verified(report_path,'text/plain')
atomic_json(WORK/'validation.json',REPORT)
atomic_json(DURABLE/'entrega_verificada.json',{'files':links+[report_link],'validation':REPORT})
print('ENTREGA CARGADA Y VERIFICADA EN DRIVE PRIVADO')
from IPython.display import display,Markdown
for item in links+[report_link]:display(Markdown('['+item['name']+']('+item['webViewLink']+')'))
print('Carpeta: https://drive.google.com/drive/folders/'+remote_folder['id'])
print('Android: abrí el MP4 en Drive → menú ⋮ → Descargar.')
print('iPhone: Drive → menú ⋮ → Enviar una copia/Abrir en → Guardar en Archivos o Guardar video.')
print('El proceso verificó la carga en Drive; la descarga en el celular y la revisión editorial siguen pendientes.')
